In [24]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import os

PROJECT_ROOT = '/content/drive/Shareddrives/surgical_rehab'
DATA_ZIP_PATH = os.path.join(PROJECT_ROOT, 'data/jigsaws_preprocessed.zip')
DATA_DIR = '/content/'

!unzip -qn {DATA_ZIP_PATH} -d {DATA_DIR}

%cd {PROJECT_ROOT}

/content/drive/Shareddrives/surgical_rehab


In [26]:
!pip install -q seaborn scikit-learn scrapbook

# Imports

In [27]:
import datetime
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time
import pandas as pd
import scrapbook as sb
from sklearn.metrics import classification_report

In [28]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU")

else:
    device = torch.device("cpu")
    print("Using CPU")

Using GPU: Tesla T4


# Configurations

In [29]:
date = datetime.datetime.now().strftime("%Y-%m-%d/%H:%M")


In [30]:
TASKS = ['Suturing', 'Needle_Passing', 'Knot_Tying']
RUN_NAME = 'all_tasks'

# Configuration
NUM_SAMPLES = 1000
NUM_SURGEONS = 8

#Gesture generation
GESTURE_VOCAB_SIZE = 16  # 15 gestures + 1 MASK
MASK_TOKEN_ID = 15

#Phase generation
# PHASE_VOCAB_SIZE = 7  # 6 phases + 1 mask
# MASK_TOKEN_ID = 6

SEQ_LEN = 5
HIDDEN_DIM = 2048
VISION_DIM = 1000
LANG_DIM = 768
NUM_TIMESTEPS = 10
EPOCHS = 25
BATCH_SIZE = 32
LR = 1e-4
SEED = 42

In [31]:
temp = len(str(LR).split('.')[-1])

In [32]:
TRAIN_PATHS = {task: os.path.join(DATA_DIR, f"jigsaws_train_{task}.npz") for task in TASKS}
TEST_PATHS = {task: os.path.join(DATA_DIR, f"jigsaws_test_{task}.npz") for task in TASKS}

MODEL_SAVE_PATH = os.path.join(PROJECT_ROOT, f"outputs/models/{date}/gesture_model_{RUN_NAME}_{temp}_{BATCH_SIZE}_{HIDDEN_DIM}_{EPOCHS}.pt")
CSV_SAVE_PATH = os.path.join(PROJECT_ROOT, f"outputs/training_records/{date}/gesture_training_record_{RUN_NAME}_{temp}_{BATCH_SIZE}_{HIDDEN_DIM}_{EPOCHS}.csv")
RESULTS_SAVE_PATH = os.path.join(PROJECT_ROOT, f"outputs/training_records/{date}/gesture_results_{RUN_NAME}_{temp}_{BATCH_SIZE}_{HIDDEN_DIM}_{EPOCHS}.csv")

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(CSV_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(RESULTS_SAVE_PATH), exist_ok=True)

missing_paths = [path for path in list(TRAIN_PATHS.values()) + list(TEST_PATHS.values()) if not os.path.exists(path)]
if missing_paths:
    raise FileNotFoundError("Missing expected JIGSAWS preprocessed files:\n" + "\n".join(missing_paths))

In [33]:
# Datastructure to store hyperparameters and results
metadata = {
    'run_name': RUN_NAME,
    'tasks': TASKS,
    'date': date,
    'device': str(device),
    'num_samples': NUM_SAMPLES,
    'num_surgeons': NUM_SURGEONS,
    'gesture_vocab_size': GESTURE_VOCAB_SIZE,
    'seq_len': SEQ_LEN,
    'hidden_dim': HIDDEN_DIM,
    'vision_dim': VISION_DIM,
    'lang_dim': LANG_DIM,
    'num_timesteps': NUM_TIMESTEPS,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'seed': SEED,
}

results = []
training_epochs_record = pd.DataFrame(columns=['epoch', 'loss', 'top1_acc', 'top5_acc', 'weighted_f1', 'train_time'])

## Seed

In [34]:
# Set random seeds for reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
random.seed(SEED)
np.random.seed(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# Main

In [35]:
# Forward corruption: transition matrices
q_matrix = np.zeros((NUM_TIMESTEPS, GESTURE_VOCAB_SIZE, GESTURE_VOCAB_SIZE))
for t in range(NUM_TIMESTEPS):
    alpha = 1 - (t + 1) / NUM_TIMESTEPS
    for i in range(GESTURE_VOCAB_SIZE):
        q_matrix[t, i] = (1 - alpha) / (GESTURE_VOCAB_SIZE - 1)
        q_matrix[t, i, i] = alpha
q_matrix = torch.tensor(q_matrix, dtype=torch.float32)

def corrupt(x0, t):
    B, T = x0.shape
    probs = q_matrix[t][:, x0.view(-1)]  # (V, B*T)
    probs = probs.permute(1, 0).view(B, T, GESTURE_VOCAB_SIZE)
    xt = torch.multinomial(probs.view(-1, GESTURE_VOCAB_SIZE), 1).view(B, T)
    return xt

class PooledJIGSAWSDataset(Dataset):
    def __init__(self, task_paths):
        vision_feats = []
        lang_feats = []
        gesture_seqs = []
        surgeon_ids = []
        task_names = []

        self.task_sizes = {}
        for task, npz_path in task_paths.items():
            data = np.load(npz_path)
            num_items = len(data["surgeon_id"])
            vision_feats.append(torch.tensor(data["vision_feat"], dtype=torch.float32))
            lang_feats.append(torch.tensor(data["lang_feat"], dtype=torch.float32))
            gesture_seqs.append(torch.tensor(data["gesture_seq"], dtype=torch.long))
            surgeon_ids.append(torch.tensor(data["surgeon_id"], dtype=torch.long))
            task_names.extend([task] * num_items)
            self.task_sizes[task] = num_items

        self.vision_feat = torch.cat(vision_feats, dim=0)
        self.lang_feat = torch.cat(lang_feats, dim=0)
        self.gesture_seq = torch.cat(gesture_seqs, dim=0)
        self.surgeon_id = torch.cat(surgeon_ids, dim=0)
        self.task = task_names
        self.timesteps = torch.randint(0, NUM_TIMESTEPS, (len(self.surgeon_id),))

    def __len__(self):
        return len(self.surgeon_id)

    def __getitem__(self, idx):
        x0 = self.gesture_seq[idx]
        t = self.timesteps[idx]
        xt = corrupt(x0.unsqueeze(0), t).squeeze(0)
        return {
            "vision_feat": self.vision_feat[idx],
            "lang_feat": self.lang_feat[idx],
            "surgeon_id": self.surgeon_id[idx],
            "x0": x0,
            "xt": xt,
            "t": t,
            "task": self.task[idx]
        }

class TimeEmbedding(nn.Module):
    def __init__(self, num_timesteps, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(num_timesteps, hidden_dim)

    def forward(self, t):  # (B,)
        return self.embed(t)

class SurgeonEmbedding(nn.Module):
    def __init__(self, num_surgeons, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_surgeons, hidden_dim)

    def forward(self, surgeon_id):
        return self.embedding(surgeon_id)


class VLAEncoder(nn.Module):
    def __init__(self, vision_dim, lang_dim, hidden_dim):
        super().__init__()
        self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        self.lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)

    def forward(self, vision_feat, lang_feat):
        v = self.vision_proj(vision_feat)
        l = self.lang_proj(lang_feat)
        return self.fuse(torch.cat([v, l], dim=-1))

class DiffusionGestureModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 2*hidden_dim),
            nn.ReLU(),
            nn.Linear(2*hidden_dim, 1*hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)

        )

    def forward(self, xt, cond):
        x = self.token_embed(xt)
        cond = cond.unsqueeze(1).expand_as(x)
        return self.net(x + cond)

class FullModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vla = VLAEncoder(VISION_DIM, LANG_DIM, HIDDEN_DIM)
        self.surgeon_embed = SurgeonEmbedding(NUM_SURGEONS, HIDDEN_DIM)
        #self.surgeon_embed = SurgeonEmbedding()  # no need for num_surgeons
        self.time_embed = TimeEmbedding(NUM_TIMESTEPS, HIDDEN_DIM)
        self.denoise = DiffusionGestureModel(GESTURE_VOCAB_SIZE, HIDDEN_DIM)

    def forward(self, xt, vision_feat, lang_feat, surgeon_id, t):
        cond = self.vla(vision_feat, lang_feat) + self.surgeon_embed(surgeon_id) + self.time_embed(t)
        return self.denoise(xt, cond)


In [36]:
def train():
    train_set = PooledJIGSAWSDataset(TRAIN_PATHS)
    test_set = PooledJIGSAWSDataset(TEST_PATHS)

    print(f"Train set size: {len(train_set)}, Test set size: {len(test_set)}")
    print(f"Train task sizes: {train_set.task_sizes}")
    print(f"Test task sizes: {test_set.task_sizes}")

    expected_train_size = sum(train_set.task_sizes.values())
    expected_test_size = sum(test_set.task_sizes.values())
    assert len(train_set) == expected_train_size, "Pooled train size does not match sum of task train sizes"
    assert len(test_set) == expected_test_size, "Pooled test size does not match sum of task test sizes"

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

    sample_batch = next(iter(train_loader))
    expected_keys = {"vision_feat", "lang_feat", "surgeon_id", "x0", "xt", "t", "task"}
    assert expected_keys.issubset(sample_batch.keys()), f"Missing batch keys: {expected_keys - set(sample_batch.keys())}"

    model = FullModel()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    model.to(device)

    train_time = 0.0
    if device.type == 'cuda':
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_acc = 0.0
        train_top5 = 0.0
        all_preds = []
        all_labels = []

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            vision_feat = batch["vision_feat"].to(device)
            lang_feat = batch["lang_feat"].to(device)
            surgeon_id = batch["surgeon_id"].to(device)
            x0 = batch["x0"].to(device)
            xt = batch["xt"].to(device)
            t = batch["t"].to(device)

            if device.type == 'cuda':
                start.record()
            else:
                train_start_time = time.time()

            logits = model(xt, vision_feat, lang_feat, surgeon_id, t)
            loss = criterion(logits.view(-1, GESTURE_VOCAB_SIZE), x0.view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if device.type == 'cuda':
                end.record()
                torch.cuda.synchronize()
                train_time += start.elapsed_time(end) / 1000.0
            else:
                train_end_time = time.time()
                train_time += (train_end_time - train_start_time)

            with torch.no_grad():
                preds = torch.argmax(logits, dim=-1)
                correct = (preds == x0).float().mean().item()
                train_acc += correct

                top5_preds = torch.topk(logits, k=5, dim=-1).indices
                match_top5 = top5_preds.eq(x0.unsqueeze(-1)).any(dim=-1).float().mean().item()
                train_top5 += match_top5

                all_preds.extend(preds.view(-1).cpu().numpy())
                all_labels.extend(x0.view(-1).cpu().numpy())

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_train_acc = train_acc / len(train_loader)
        avg_train_top5 = train_top5 / len(train_loader)
        weighted_f1 = classification_report(all_labels, all_preds, labels=list(range(GESTURE_VOCAB_SIZE)), output_dict=True, zero_division=0)['weighted avg']['f1-score']

        print(f"Epoch {epoch+1} Loss: {avg_train_loss:.4f} | Top-1 Acc: {avg_train_acc:.4f} | Top-5 Acc: {avg_train_top5:.4f}  | Weighted F1: {weighted_f1:.4f} | Train Time: {train_time:.2f}s")
        training_epochs_record.loc[len(training_epochs_record)] = [epoch+1, avg_train_loss, avg_train_acc, avg_train_top5, weighted_f1, train_time]

    metadata['model'] = str(model)

    model.eval()
    with torch.no_grad():
        test_loss = 0.0
        test_acc = 0.0
        test_top5 = 0.0
        all_preds = []
        all_labels = []
        all_top5_matches = []
        all_task_labels = []

        for batch in tqdm(test_loader, desc="Testing", leave=False):
            vision_feat = batch["vision_feat"].to(device)
            lang_feat = batch["lang_feat"].to(device)
            surgeon_id = batch["surgeon_id"].to(device)
            x0 = batch["x0"].to(device)
            xt = batch["xt"].to(device)

            B, L = xt.shape
            logits = None

            for step in range(NUM_TIMESTEPS):
                time_step = NUM_TIMESTEPS - 1 - step
                t_step_tensor = torch.full((B,), time_step, dtype=torch.long, device=device)
                logits = model(xt, vision_feat, lang_feat, surgeon_id, t_step_tensor)
                probs = torch.softmax(logits, dim=-1)
                xt = torch.multinomial(probs.view(-1, GESTURE_VOCAB_SIZE), 1).view(B, L)

            loss = criterion(logits.view(-1, GESTURE_VOCAB_SIZE), x0.view(-1))
            preds = torch.argmax(logits, dim=-1)
            correct = (preds == x0).float().mean().item()
            test_acc += correct

            top5_preds = torch.topk(logits, k=5, dim=-1).indices
            top5_matches = top5_preds.eq(x0.unsqueeze(-1)).any(dim=-1)
            match_top5 = top5_matches.float().mean().item()
            test_top5 += match_top5

            all_preds.extend(preds.view(-1).cpu().numpy())
            all_labels.extend(x0.view(-1).cpu().numpy())
            all_top5_matches.extend(top5_matches.view(-1).cpu().numpy())
            for task in batch["task"]:
                all_task_labels.extend([task] * L)

            test_loss += loss.item()

        avg_test_loss = test_loss / len(test_loader)
        all_preds_np = np.array(all_preds)
        all_labels_np = np.array(all_labels)
        all_top5_matches_np = np.array(all_top5_matches, dtype=bool)
        all_task_labels_np = np.array(all_task_labels)

        avg_test_acc = float(np.mean(all_preds_np == all_labels_np))
        avg_test_top5 = float(np.mean(all_top5_matches_np))
        test_weighted_f1 = classification_report(all_labels, all_preds, labels=list(range(GESTURE_VOCAB_SIZE)), output_dict=True, zero_division=0)['weighted avg']['f1-score']

        print(f"\nCombined Test Loss: {avg_test_loss:.4f} | Top-1 Acc: {avg_test_acc:.4f} | Top-5 Acc: {avg_test_top5:.4f} | Weighted F1: {test_weighted_f1:.4f}")
        print(classification_report(all_labels, all_preds, labels=list(range(GESTURE_VOCAB_SIZE)), zero_division=0))

        results.append({
            'kind': 'Gesture',
            'scope': 'combined',
            'task': RUN_NAME,
            'type': 'from scratch',
            'loss': avg_test_loss,
            'accuracy': avg_test_acc,
            'top5_accuracy': avg_test_top5,
            'weighted_f1': test_weighted_f1,
            'num_sequences': len(test_set),
            'num_tokens': len(all_labels),
            'train_time': train_time,
            'hidden_dim': HIDDEN_DIM,
            'batch_size': BATCH_SIZE,
            'lr': LR
        })

        print("\nPer-task evaluation:")
        for task in TASKS:
            task_mask = all_task_labels_np == task
            task_labels = all_labels_np[task_mask]
            task_preds = all_preds_np[task_mask]
            task_top5 = all_top5_matches_np[task_mask]
            task_acc = float(np.mean(task_preds == task_labels)) if len(task_labels) else 0.0
            task_top5_acc = float(np.mean(task_top5)) if len(task_top5) else 0.0
            task_weighted_f1 = classification_report(task_labels, task_preds, labels=list(range(GESTURE_VOCAB_SIZE)), output_dict=True, zero_division=0)['weighted avg']['f1-score'] if len(task_labels) else 0.0
            task_sequences = int(test_set.task_sizes[task])
            task_tokens = int(len(task_labels))

            print(f"{task}: Top-1 Acc: {task_acc:.4f} | Top-5 Acc: {task_top5_acc:.4f} | Weighted F1: {task_weighted_f1:.4f} | Sequences: {task_sequences}")
            results.append({
                'kind': 'Gesture',
                'scope': 'task',
                'task': task,
                'type': 'from scratch',
                'loss': np.nan,
                'accuracy': task_acc,
                'top5_accuracy': task_top5_acc,
                'weighted_f1': task_weighted_f1,
                'num_sequences': task_sequences,
                'num_tokens': task_tokens,
                'train_time': train_time,
                'hidden_dim': HIDDEN_DIM,
                'batch_size': BATCH_SIZE,
                'lr': LR
            })

    return model, train_time


## Training

In [37]:
model, train_time = train()
torch.save(model.state_dict(), MODEL_SAVE_PATH)

Train set size: 18159, Test set size: 4541
Train task sizes: {'Suturing': 8403, 'Needle_Passing': 5816, 'Knot_Tying': 3940}
Test task sizes: {'Suturing': 2101, 'Needle_Passing': 1454, 'Knot_Tying': 986}


Epoch 1 Loss: 1.0396 | Top-1 Acc: 0.6420 | Top-5 Acc: 0.9622  | Weighted F1: 0.6380 | Train Time: 8.45s


Epoch 2 Loss: 0.7628 | Top-1 Acc: 0.7239 | Top-5 Acc: 0.9888  | Weighted F1: 0.7226 | Train Time: 16.79s


Epoch 3 Loss: 0.6552 | Top-1 Acc: 0.7597 | Top-5 Acc: 0.9939  | Weighted F1: 0.7588 | Train Time: 25.17s


Epoch 4 Loss: 0.5733 | Top-1 Acc: 0.7893 | Top-5 Acc: 0.9965  | Weighted F1: 0.7889 | Train Time: 33.71s


...

Epoch 22 Loss: 0.1511 | Top-1 Acc: 0.9472 | Top-5 Acc: 1.0000  | Weighted F1: 0.9471 | Train Time: 184.91s


Epoch 23 Loss: 0.1340 | Top-1 Acc: 0.9516 | Top-5 Acc: 1.0000  | Weighted F1: 0.9516 | Train Time: 193.31s


Epoch 24 Loss: 0.1344 | Top-1 Acc: 0.9518 | Top-5 Acc: 1.0000  | Weighted F1: 0.9518 | Train Time: 201.75s


Epoch 25 Loss: 0.1315 | Top-1 Acc: 0.9523 | Top-5 Acc: 1.0000  | Weighted F1: 0.9523 | Train Time: 210.22s



Combined Test Loss: 1.4847 | Top-1 Acc: 0.7739 | Top-5 Acc: 0.9976 | Weighted F1: 0.7733
              precision    recall  f1-score   support

           0       0.75      0.67      0.71       579
           1       0.80      0.78      0.79      4062
           2       0.82      0.82      0.82      4598
           3       0.71      0.75      0.73      1922
           4       0.65      0.65      0.65       664
           5       0.70      0.76      0.73      3047
           6       0.00      0.00      0.00         0
           7       0.78      0.74      0.76      1240
           8       0.76      0.71      0.73       204
           9       0.81      0.88      0.84       114
          10       0.86      0.73      0.79      1391
          11       0.81      0.80      0.80       969
          12       0.83      0.54      0.65       661
          13       0.72      0.83      0.77       924
          14       0.81      0.91      0.86      1455
          15       0.73      0.76      0.74  

In [38]:
print(f"train time: {train_time}")

# sb.glue('metadata', metadata)
# sb.glue('results', results)

training_epochs_record.to_csv(CSV_SAVE_PATH, index=False)
pd.DataFrame(results).to_csv(RESULTS_SAVE_PATH, index=False)

train time: 210.22231442642135
